In [0]:
path="development_042_silver_sandbox.demand_forecast.eilers_to_gdna_mapping_bronze"
dest="development_042_silver_sandbox.demand_forecast.eilers_to_gdna_mapping_silver"

In [0]:
df = spark.read.table(path)
display(df.describe())

In [0]:
from pyspark.sql.functions import col, when, isnull

df_updated = df.withColumn(
    "correction",
    when(col("correction").contains("?"), "no_match")
    .when(col("correction") == "no match", "no_match")
    .when(isnull(col("correction")), col("ep_theme_name"))
    .otherwise(col("correction"))
)
TOLOWER=['bp_theme_name','ep_theme_name','correction']
df_lower = df_updated.select([lower(col(c)).alias(c) for c in TOLOWER])
display(df_lower)

In [0]:
no_match_count = df_lower.filter(col("correction") == "no_match").count()
total_count = df_lower.count()
percentage = (no_match_count / total_count) * 100

print(f"No_match count: {no_match_count}")
print(f"Percentage of no_match: {percentage:.2f}%")

In [0]:
df_lower.write.mode("overwrite").saveAsTable(dest)